# Question 15 - 135. Candy

There are `n` children standing in a line. Each child is assigned a rating value given in the integer array `ratings`.

You are giving candies to these children subjected to the following requirements:

- Each child must have at least one candy.
- Children with a higher rating get more candies than their neighbors.

Return *the **minimum** number of candies you need to have to distribute the candies to the children*.

**Example 1:**

    Input: ratings = [1,0,2]
    Output: 5
    Explanation: You can allocate to the first, second and third child with 2, 1, 2 candies respectively.

**Example 2:**

    Input: ratings = [1,2,2]
    Output: 4
    Explanation: You can allocate to the first, second and third child with 1, 2, 1 candies respectively.
    The third child gets 1 candy because it satisfies the above two conditions.

**Constraints:**

- `n == ratings.length`
- `1 <= n <= 2 * 10^4`
- `0 <= ratings[i] <= 2 * 10^4`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two-pass greedy (satisfy left neighbours, then right neighbours)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (relax until stable) | O(n²) | O(n) |
| 2️⃣ Optimized (two passes) | O(n) | O(n) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Every child gets at least 1. If my rating is strictly higher than a neighbour's, I must get strictly more candy than that neighbour. I want the minimum total."*

- **Equal** ratings: any constraint? → **No.** Equal neighbours can get any amounts (see Example 2: `[1,2,2]` → `1,2,1`). People often get this wrong.
- Only immediate neighbours matter (left and right)? → **Yes.**
- Return the total count, not the distribution.

### Step 1 · Brute force

**Idea:** Start everyone at 1. Repeatedly sweep the line and fix any violation (`ratings[i] > ratings[i±1]` but `candies[i] <= candies[i±1]` → bump `candies[i]` to neighbour + 1). Stop when a full sweep changes nothing.

🗣️ *"The simplest thing that's obviously correct is to keep fixing violations until there are none. Each fix only raises a value to the smallest number that satisfies a constraint, so we never overshoot and the result is minimal. But a long decreasing run like `[5,4,3,2,1]` needs about n sweeps to push the values back, so it's O(n²)."*

In [1]:
class SolutionBrute:
    def candy(self, ratings: list[int]) -> int:
        n = len(ratings)
        candies = [1] * n
        changed = True
        while changed:
            changed = False
            for i in range(n):
                if i > 0 and ratings[i] > ratings[i - 1] and candies[i] <= candies[i - 1]:
                    candies[i] = candies[i - 1] + 1
                    changed = True
                if i < n - 1 and ratings[i] > ratings[i + 1] and candies[i] <= candies[i + 1]:
                    candies[i] = candies[i + 1] + 1
                    changed = True
        return sum(candies)

### Step 2 · Optimize

**Bottleneck:** Each child has **two** constraints (left and right), and fixing one can break the other, so brute force goes back and forth repeatedly.

**Key insight: handle the two constraints separately, each in the direction where it can be satisfied in a single pass.**

- **Left → right pass:** only enforce *"higher than left neighbour ⇒ more than left"*. Going left to right, the left neighbour is already final for this rule, so `left[i] = left[i−1] + 1` if rating rises, else `1`.
- **Right → left pass:** only enforce *"higher than right neighbour ⇒ more than right"*. Symmetric.
- **Combine:** child `i` needs **both**, so it gets `max(left[i], right[i])`. Taking the max satisfies both rules, and it's the minimum because each pass already gave the smallest value for its own rule.

In code, I reuse one array: do the left pass, then during the right pass take `max(candies[i], candies[i+1] + 1)`.

🗣️ *"Each child has a left constraint and a right constraint. A left-to-right pass handles all left constraints in one go, a right-to-left pass handles all right constraints, and taking the max at each position satisfies both with the fewest candies. Two linear passes, O(n)."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Relax-until-stable | ❌ | Correct but O(n²) on long decreasing runs. |
| Sort children by rating, assign in increasing order | ⚠️ | Correct (`1 + max(lower neighbours)`), but O(n log n) because of the sort. |
| Graph + topological order | ❌ | Same idea as sorting, but heavier to build. |
| **Two passes + max** | ✅ | O(n), simple, and easy to justify. Each pass handles one direction of constraints. |
| One pass counting up/down slopes | ✅ (advanced) | O(1) extra space, but tricky peak bookkeeping. Mention it as a follow-up, don't lead with it. |

In [2]:
class Solution:
    def candy(self, ratings: list[int]) -> int:
        n = len(ratings)
        candies = [1] * n
        # pass 1: satisfy "more than LEFT neighbour if higher rating"
        for i in range(1, n):
            if ratings[i] > ratings[i - 1]:
                candies[i] = candies[i - 1] + 1
        # pass 2: satisfy "more than RIGHT neighbour" without breaking pass 1
        for i in range(n - 2, -1, -1):
            if ratings[i] > ratings[i + 1]:
                candies[i] = max(candies[i], candies[i + 1] + 1)
        return sum(candies)

### Step 3 · Toy example walkthrough

`ratings = [1, 3, 4, 5, 2]` (a peak at index 3)

| | i=0 | i=1 | i=2 | i=3 | i=4 |
|---|---|---|---|---|---|
| ratings | 1 | 3 | 4 | 5 | 2 |
| start | 1 | 1 | 1 | 1 | 1 |
| after L→R (rising?) | 1 | 2 | 3 | **4** | 1 |
| R→L candidate `right[i+1]+1` if ratings[i] > ratings[i+1] | n/a | n/a | n/a | 5>2 → 2 | n/a |
| after max | 1 | 2 | 3 | max(4,2)=**4** | 1 |

Total = 1+2+3+4+1 = **11**.

Why `max` and not overwrite? The peak at index 3 needed **4** from the left side (it sits on a staircase 1,2,3). The right side only asks for 2. Overwriting with 2 would break `5 > 4 ⇒ more than index 2`.

**Example 2:** `[1, 2, 2]` → L→R: `[1, 2, 1]` (equal ratings → reset to 1). R→L: nothing changes. Total **4** ✅

In [3]:
cases = [
    ([1, 0, 2], 5),
    ([1, 2, 2], 4),
    ([1], 1),
    ([1, 3, 4, 5, 2], 11),
    ([5, 4, 3, 2, 1], 15),
    ([1, 2, 3, 4, 5], 15),
    ([2, 2, 2], 3),
    ([1, 3, 2, 2, 1], 7),
]
for ratings, expected in cases:
    assert SolutionBrute().candy(ratings) == expected
    assert Solution().candy(ratings) == expected
    print(ratings, "->", expected)

import random
for _ in range(500):
    r = [random.randint(0, 4) for _ in range(random.randint(1, 10))]
    assert Solution().candy(r) == SolutionBrute().candy(r), r
print("All tests passed ✅")

[1, 0, 2] -> 5
[1, 2, 2] -> 4
[1] -> 1
[1, 3, 4, 5, 2] -> 11
[5, 4, 3, 2, 1] -> 15
[1, 2, 3, 4, 5] -> 15
[2, 2, 2] -> 3
[1, 3, 2, 2, 1] -> 7
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Relax until stable | O(n²) | O(n) | Up to n sweeps on monotone runs |
| Sort by rating | O(n log n) | O(n) | Sorting dominates |
| **Two passes** | **O(n)** | **O(n)** | Two linear passes plus a sum |
| Slope counting | O(n) | O(1) | Counts up/down run lengths, with tricky peak handling |

**Complexity, explained:** three linear passes (left, right, sum), each O(n). The `candies` array is O(n) space.

**Edge cases:** equal neighbours (no constraint, reset to 1), strictly decreasing (n(n+1)/2), single child (1).

**Follow-up:** *O(1) space?* → Walk the array tracking the current `up` run length, `down` run length, and the `peak`. A down run of length `d` costs `1+2+…+d`, and the peak gets bumped to `max(up, down)+1`. Mention it, but code it only if asked.

🗣️ **30-second recap:** *"Each child has a left-neighbour rule and a right-neighbour rule. One left-to-right pass gives the minimum that satisfies all left rules, one right-to-left pass does the same for right rules, and taking the max at each child satisfies both with the fewest candies. O(n) time and O(n) space. There's an O(1)-space slope-counting version if needed."*